# 02. Python 데이터 구조 — 수업 핵심 60분

> **수업 핵심:** 리스트·튜플·딕셔너리와 2단계 센서 보고서
>
> **선택/자습 확장:** `deque` 큐와 서점 주문 처리기
>
> **환경:** Python 3 / JupyterLab · 표준 라이브러리만 사용

Python 프로그램은 데이터를 **어떤 모양으로 담고, 어떤 이름으로 찾고, 어떻게 바꿀지** 정하는 일에서 시작합니다.

## 학습 목표

### 수업 핵심 60분

- 리스트의 인덱싱·슬라이싱·추가·복사를 설명하고 원본을 보존하며 처리한다.
- 튜플을 언패킹하고 이름 있는 함수를 `max(..., key=...)`의 비교 기준으로 전달한다.
- 딕셔너리의 키로 값을 안전하게 조회하고 설비별 데이터를 누적한다.
- 센서 데이터를 **그룹 만들기 → 요약 만들기** 두 단계로 나누어 구현한다.
- 결과 딕셔너리를 JSON으로 저장하고 다시 읽어 확인한다.

### 선택/자습 확장

- `collections.deque`로 FIFO 큐를 구현한다.
- 리스트·튜플·딕셔너리·큐를 서점 주문 처리 문제에서 함께 사용한다.

각 코드 셀은 <kbd>Shift</kbd>+<kbd>Enter</kbd>로 실행합니다. “예측하기”가 나오면 실행 전에 결과를 먼저 적고, TODO는 강사가 안내하는 시점에 정답과 비교하고 자신의 코드를 다시 검증하세요.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 이 노트북만으로 시작하기

열 파일은 `/workspace/02.Python/02_data_structures.ipynb`입니다. [01_python_basics.ipynb의 「반복문」·「함수」](01_python_basics.ipynb)에 앞선 설명이 있으며 필요한 내용은 다음과 같습니다. `for value in values:`는 원소를 하나씩 읽고, `if 조건:`은 참일 때만 블록을 실행하며, `def 이름(입력):` 안의 `return 결과`는 다른 셀이 사용할 값을 돌려줍니다. `print`는 화면 표시일 뿐 반환값을 대신하지 않습니다.

이전 노트북 변수·파일은 필요하지 않습니다. 현재 첫 환경 셀이 `OUTPUT=/workspace/Data/outputs/02.Python`을 만들고, 「0. 한눈에 보는 선택 기준」 아래 코드 셀이 `section` 도우미를 정의합니다. 핵심 경로는 리스트→튜플→딕셔너리→센서 1·2단계→저장이며 입력 측정값은 현재 노트북에 포함되어 있습니다. 저장 결과는 `/workspace/Data/outputs/02.Python/sensor_report.json`입니다.
<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

여러 센서 기록을 한 번에 처리하려면 저장 형태부터 정해야 합니다. 리스트는 순서 있는 기록, 튜플은 한 기록의 관련 값, 딕셔너리는 설비 이름으로 찾는 묶음입니다. 마지막 보고서는 이 자료구조를 연결해 원본을 보존하면서 설비별 상태를 설명하는 결과입니다.

### 전체 과정에서 내 코드가 하는 일

```text
원본 기록
  ↓
설비별 그룹
  ↓
개수·평균·경보 요약
  ↓
JSON 저장
  ↓
다시 읽어 내용 비교
```

그룹 생성과 요약 함수를 나누어 작성합니다. max의 key 함수는 비교할 점수를 내지만 max는 원본 레코드를 반환합니다. 중간 평균을 반복해서 반올림하지 말고 최종 표현 단계의 약속을 지킵니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 평균이 임계값 아래인데 경보가 켜질 수 있나요?

**설명:** 그럴 수 있습니다. 해당 과제의 경보는 개별 측정값 중 임계값 이상이 있는지로 판단합니다. 평균·최댓값·경보는 서로 다른 질문입니다. 보고서의 숫자가 왜 나왔는지 원본 기록까지 거슬러 설명해야 합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| `NameError` 또는 TODO 검사 실패 | 환경·정의 셀을 먼저 실행하고 TODO의 반환값/경계 조건을 정답 해설과 비교합니다. | 초기화 성공과 학습자 검사 통과 |
| `KeyError` / `IndexError` | 주문 품목 키와 빈 큐 여부를 확인합니다. pop/popleft 전에 입력과 상태를 점검합니다. | 빈 입력·없는 품목 처리 검사 통과 |
| 큐가 줄지 않음·합계 중복 | 같은 객체 재사용과 루프의 상태 갱신을 확인합니다. 초기 데이터 셀에서 새 상태로 비교합니다. | 처리 순서·재고·합계가 정답과 일치 |

**예시에 없는 오류를 확인할 곳:** deque·서점 주문 확장의 검증 셀과 정답 설명, [Python README](README.md).


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **리스트·튜플·딕셔너리와 제조 센서 보고서까지** → [1일차 슬라이드 54쪽](https://docs.google.com/presentation/d/1pQvfu3XzIfFL9FFvJkP8a6iNKJMjQFGOQIboEF6DRjg/edit#slide=id.d1_s049) · [배포 PDF 54쪽](../슬라이드/1일차.pdf#page=54)에서 65쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 설비별 통계와 JSON을 저장합니다.
- **단원 마지막 연결 질문:** 평균이 같아도 최고 온도나 누락값이 다른 두 설비를 같은 상태라고 판단해도 될까요? 단위와 누락 여부를 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


## 이번 과정의 실행 안내

**수업 핵심 / 60분.** 본인 Docker 컨테이너의 Python 커널에서 실행합니다. GUI·외부 패키지가 필요하지 않습니다.
예제→TODO→학습자 검증→접힌 정답→정답 검증 순서입니다. 미완성 TODO 검증 실패는 정상입니다.
정답은 별도 변수·함수 이름을 사용합니다. 정답을 실행해도 자신의 TODO가 완성되지는 않습니다.
학습자 검증을 통과한 뒤 다음 절로 진행하세요. 노트북을 저장하고 종료하면 호스트 복사본에 남습니다.


## 프로그래밍이 처음이라면 먼저 읽으세요

프로그램은 컴퓨터가 수행할 일을 정해진 문법으로 적은 **명령의 모음** 입니다. Python에서는 사람이 쓴 코드를 **Python 인터프리터** 가 위에서 아래로 읽어 실행합니다. 이 노트북의 **커널(kernel)** 은 그 인터프리터를 계속 실행해 두는 작은 작업 공간입니다. 한 셀에서 만든 변수와 함수가 다음 셀에서도 보이는 이유가 바로 커널이 상태를 기억하기 때문입니다.

처음에는 다음 용어만 구분하면 충분합니다.

| 용어 | 쉬운 뜻 | 예 |
|---|---|---|
| 값(value) | 프로그램이 다루는 실제 데이터 | `3`, `2.5`, `"Jetson"`, `True` |
| 변수(variable) | 값을 다시 찾기 위해 붙인 이름 | `temperature = 23.5` |
| 표현식(expression) | 계산되어 하나의 값이 되는 코드 | `temperature >= 30` |
| 문장(statement) | 대입·분기·반복처럼 수행할 일을 지시하는 코드 | `if alarm:` |
| 함수(function) | 입력을 받아 정해진 처리를 재사용하는 코드 묶음 | `len(values)` |
| 객체(object) | 값과 그 값이 할 수 있는 동작을 함께 보는 단위 | `values.append(3)`의 `values` |
| 오류/예외(error/exception) | 실행할 수 없는 이유를 알려 주는 진단 정보 | `NameError`, `TypeError` |

### 처음 보는 기호 읽기

| 기호 | 역할 | 읽는 예 |
|---|---|---|
| `# ...` | 그 줄의 설명인 주석. 실행되지 않음 | `# 온도 저장` |
| `=` | 오른쪽 값을 왼쪽 이름에 대입 | `x = 3` |
| `==` | 두 값이 같은지 비교 | `x == 3` |
| `()` | 함수 호출, 연산 순서 묶기, 튜플 표현 | `print(x)` |
| `[]` | 리스트 표현 또는 위치로 항목 선택 | `items[0]` |
| `{}` | 딕셔너리 표현 | `{"name": "A"}` |
| `:` | 조건·반복·함수·클래스 블록의 시작 | `if ready:` |
| 들여쓰기 | 어느 블록에 속한 문장인지 표시 | 보통 공백 4칸 |

코드의 `# TODO:`는 학습자가 채울 위치이고, `pass`는 아직 아무 동작도 하지 않는 빈 블록을 문법적으로 유지하는 문장입니다. `None`은 “아직 유효한 결과가 없음”을 나타내는 특별한 값으로 starter 코드에 사용됩니다.

### 노트북 셀을 안전하게 실행하는 습관

1. Markdown 셀에서 **실행 전 예측** 을 먼저 읽습니다.
2. 코드 셀을 선택하고 <kbd>Shift</kbd>+<kbd>Enter</kbd>를 한 번 누릅니다.
3. 셀 왼쪽의 `[*]`가 숫자로 바뀔 때까지 기다립니다. `[*]`는 실행 중이라는 뜻입니다.
4. 출력과 예상이 다르면 `print(...)`로 중간값을 확인합니다.
5. TODO를 수정한 뒤에는 **TODO 셀부터 다시 실행** 하고 검증 셀을 실행합니다.

검증 셀의 `assert 조건, "설명"`은 조건이 참이면 조용히 다음 줄로 진행하고, 거짓이면 `AssertionError`와 설명을 보여 줍니다. 마지막에 `learner_verification=PASS`가 출력되어야 자신의 TODO가 검증된 것입니다. 검증 셀의 기대값을 바꾸면 학습 목표 자체가 달라지므로 검증 코드는 수정하지 않습니다.

셀 왼쪽 실행 번호가 `In [8]`, `In [3]`처럼 뒤섞이면 코드가 화면의 위→아래 순서와 다르게 실행된 상태입니다. 이해하기 어려운 결과가 나오면 JupyterLab 메뉴에서 **Kernel → Restart Kernel and Run All Cells** 를 선택하되, 미완성 TODO 검증은 실패하는 것이 정상임을 기억하세요. 커널 재시작은 메모리의 변수만 지우며 저장한 노트북 파일과 `Data` 파일은 지우지 않습니다.

### 오류 메시지를 읽는 순서

빨간 글씨가 길어도 마지막 줄부터 읽으면 됩니다.

| 마지막 줄 | 주된 원인 | 먼저 확인할 것 |
|---|---|---|
| `NameError` | 이름을 만들기 전에 사용함 | 앞 셀 실행 여부, 철자와 대소문자 |
| `TypeError` | 맞지 않는 타입끼리 연산함 | `type(...)` 결과, 문자열과 숫자 혼합 여부 |
| `SyntaxError` | Python 문법에 맞지 않음 | 빠진 `:`, 괄호·따옴표 짝 |
| `IndentationError` | 들여쓰기 구조가 잘못됨 | `if`, `for`, `def`, `class` 아래 공백 4칸 |
| `AssertionError` | 검증에서 기대한 결과와 다름 | TODO 셀 재실행 여부와 검증 메시지 |

오류는 실패 판정이 아니라 **어느 줄에서 어떤 가정이 어긋났는지 알려 주는 자료** 입니다. 마지막 줄의 오류 이름을 읽고, 그 위에서 자신의 코드가 표시된 가장 가까운 줄을 찾으세요.


In [ ]:
from pathlib import Path
import os, sys
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "02.Python").is_dir() and (p / "03.C++").is_dir()), None)
assert root is not None, "강의 작업 폴더에서 실행하세요."
DATA = root / "Data"
OUTPUT = DATA / "outputs" / "02.Python"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable, "결과:", OUTPUT)


### 환경 셀 출력 읽기

이 셀은 실습 답을 계산하지 않습니다. 현재 위치에서 강의 루트와 결과 저장 폴더를 찾아 이후 셀이 같은 경로를 쓰게 준비합니다.

- `Python: /opt/course/bin/python`이 보이면 강의 컨테이너의 Python을 사용하고 있습니다.
- `결과: .../Data/outputs/02.Python`은 저장 파일이 만들어질 위치입니다.
- `강의 작업 폴더에서 실행하세요`라는 오류가 나오면 JupyterLab에서 강의자료 복사본 안의 노트북을 열었는지 확인하세요.

뒤 과목에서도 같은 원칙을 씁니다. 모델 체크포인트, 카메라 처리 결과, ROS 로그처럼 컨테이너를 지운 뒤에도 필요한 것은 `Data` 아래에 둡니다.


## 60분 진행표

| 시간 | 수업 핵심 |
|---|---|
| 0–5분 | 환경·자료구조 선택 기준 확인 |
| 5–18분 | 리스트·복사·기온 분석 함수 |
| 18–28분 | 튜플·언패킹·이름 있는 `key` 함수 |
| 28–40분 | 딕셔너리 조회·순회·빈도 누적 |
| 40–47분 | 센서 1단계: 설비별 값 그룹 만들기 |
| 47–56분 | 센서 2단계: 통계 요약 만들기 |
| 56–60분 | JSON 저장·핵심 완료 점검 |

센서 실습 다음의 큐와 서점 주문 처리기는 **선택/자습 확장 20–30분** 입니다. 수업에서는 센서 결과를 저장한 뒤 선택 절을 건너뛰어 맨 끝의 전체 마무리로 이동합니다.


## 0. 한눈에 보는 선택 기준

| 구조 | 순서 | 변경 가능 | 중복 | 대표 접근 | 잘 맞는 상황 |
|---|---|---|---|---|---|
| 리스트 `list` | 있음 | 가능 | 가능 | 정수 인덱스 | 순서 있는 항목을 모으고 수정할 때 |
| 튜플 `tuple` | 있음 | 불가능 | 가능 | 정수 인덱스 | 바뀌면 안 되는 묶음, 고정 형식 레코드 |
| 딕셔너리 `dict` | 삽입 순서 유지 | 가능 | 키는 중복 불가 | 키 | 이름표로 값을 빠르게 찾을 때 |
| 덱 `deque` | 있음 | 가능 | 가능 | 양 끝 | 앞에서 넣고 빼는 큐·덱 |

> Python 3.7 이상에서 딕셔너리는 삽입 순서를 유지하지만, 딕셔너리의 핵심은 **위치가 아니라 키로 찾기** 입니다.

### 먼저 생각하기

- 할 일 여러 개를 순서대로 저장하고 중간 항목도 수정한다 → **리스트**
- 위도와 경도처럼 두 값의 형식을 고정한다 → **튜플**
- 상품명으로 재고를 찾는다 → **딕셔너리**
- 먼저 온 고객부터 처리한다 → **큐(`deque`)**

### 자료구조가 필요한 이유

변수 하나에는 보통 값 하나를 연결합니다. 실제 프로그램은 수백 개 센서값, 여러 물체의 좌표, 모델 설정처럼 서로 관련된 값을 함께 다뤄야 합니다. **자료구조(data structure)** 는 값을 어떤 규칙으로 묶고 찾고 바꿀지 정한 그릇입니다.

구조를 고를 때 이름부터 외우기보다 세 질문을 사용하세요.

1. 항목의 **순서** 가 의미 있는가?
2. 실행 중 항목을 **추가·삭제·수정** 해야 하는가?
3. 위치보다 상품명·설비명 같은 **키** 로 찾아야 하는가?

리스트·튜플·딕셔너리는 서로 경쟁하는 정답이 아닙니다. 한 프로그램 안에서 역할에 맞게 겹쳐 사용합니다. 필수 센서 보고서에서는 입력 전체는 리스트, 측정 한 건은 튜플, 설비별 결과는 딕셔너리로 표현합니다.

### 공통 용어

| 용어 | 뜻 |
|---|---|
| 항목(element/item) | 구조 안에 들어 있는 값 하나 |
| 인덱스(index) | 순서 있는 구조의 위치 번호. Python은 0부터 시작 |
| 길이(length) | 들어 있는 항목 수. `len(...)`으로 확인 |
| 가변(mutable) | 만든 뒤 내부 항목을 바꿀 수 있음 |
| 불변(immutable) | 만든 뒤 내부 항목을 다른 값으로 바꿀 수 없음 |
| 순회(iteration) | `for`로 항목을 하나씩 방문 |
| 중첩(nesting) | 리스트 안 딕셔너리처럼 구조 안에 다른 구조를 넣음 |


In [ ]:
# 이번 노트북에서 출력 구역을 구분할 작은 도우미
def section(title):
    print(f"\n--- {title} ---")


section("준비 확인")
sample = ["리스트", ("튜플", 2), {"딕셔너리": 3}]
print("Python 데이터 구조를 사용할 준비가 되었습니다.")
print("sample의 타입:", type(sample).__name__)

## 1. 리스트(list): 순서가 있고 바꿀 수 있는 모음

리스트는 대괄호 `[]`로 만들며 항목의 순서를 기억합니다.

- 인덱스는 **0부터** 시작합니다.
- 음수 인덱스 `-1`은 마지막 항목입니다.
- 슬라이스 `start:stop`에서 `stop` 위치는 **포함하지 않습니다** .
- 항목 추가·교체·삭제가 가능하므로 **가변(mutable)** 자료형입니다.

In [ ]:
section("리스트 생성·인덱싱·슬라이싱")

fruits = ["사과", "바나나", "귤", "포도"]

print("전체:", fruits)
print("첫 번째:", fruits[0])
print("마지막:", fruits[-1])
print("인덱스 1부터 2까지:", fruits[1:3])
print("앞의 두 개:", fruits[:2])
print("역순 복사:", fruits[::-1])

# 리스트는 변경할 수 있습니다.
fruits[1] = "딸기"
print("교체 후:", fruits)

### 리스트 예제 출력 읽기

`fruits[0]`은 첫 항목 `사과`, `fruits[-1]`은 뒤에서 첫 항목 `포도`입니다. `fruits[1:3]`은 시작 위치 1은 포함하고 끝 위치 3은 포함하지 않아 `바나나`, `귤` 두 개를 새 리스트로 만듭니다. `fruits[::-1]`도 역순으로 된 새 리스트를 만들며 원본 순서를 바꾸지 않습니다.

마지막 대입 `fruits[1] = "딸기"`는 원본 리스트의 두 번째 항목을 실제로 바꿉니다. 슬라이스로 만든 결과와 원본 변경을 구분하려면 변경 전후에 `print(fruits)`를 두 번 실행해 비교하세요.

> **후속 과목 연결:** 이미지 한 장도 픽셀의 규칙적인 모음이고, 검출 결과 여러 개는 리스트로 다룹니다. 인덱스 범위를 벗어나면 `IndexError`가 나므로 먼저 `len(...)`을 확인하는 습관이 중요합니다.


### 자주 쓰는 리스트 연산

| 목적 | 코드 | 설명 |
|---|---|---|
| 끝에 추가 | `items.append(x)` | 항목 하나를 추가 |
| 여러 항목 추가 | `items.extend(xs)` | 반복 가능한 값의 항목을 각각 추가 |
| 위치에 삽입 | `items.insert(i, x)` | 인덱스 `i` 앞에 추가 |
| 값으로 삭제 | `items.remove(x)` | 첫 번째 `x` 삭제, 없으면 `ValueError` |
| 위치로 꺼내기 | `items.pop(i)` | 삭제한 값을 반환, 기본은 마지막 |
| 새 리스트 만들기 | `[식 for x in xs if 조건]` | 리스트 컴프리헨션 |

### 흔한 함정: 같은 리스트를 함께 가리키기

`alias = original`은 복사가 아닙니다. 두 변수가 **같은 리스트 객체** 를 가리킵니다. 얕은 복사는 `original.copy()` 또는 `original[:]`를 사용합니다.

> 얕은 복사는 바깥 리스트만 새로 만듭니다. 안쪽에 중첩된 리스트까지 독립적으로 복사하려면 `copy.deepcopy`가 필요할 수 있습니다.

In [ ]:
section("추가·삭제·컴프리헨션")

numbers = [1, 2, 3]
numbers.append(4)
numbers.extend([5, 6])
removed = numbers.pop()  # 마지막 값을 꺼냄

squares_of_even = [n * n for n in numbers if n % 2 == 0]

print("변경된 numbers:", numbers)
print("꺼낸 값:", removed)
print("짝수의 제곱:", squares_of_even)

section("별칭과 복사")
original = ["A", "B"]
alias = original
copied = original.copy()

alias.append("C")
copied.append("별도")

print("original:", original)
print("alias:", alias)
print("copied:", copied)
print("original is alias:", original is alias)
print("original is copied:", original is copied)

### 리스트 변경과 복사를 단계별로 확인하기

첫 부분에서 `numbers`는 `[1, 2, 3]` → append 후 `[1, 2, 3, 4]` → extend 후 `[1, 2, 3, 4, 5, 6]` → pop 후 `[1, 2, 3, 4, 5]` 순서로 바뀝니다. `removed`에는 꺼낸 값 6이 남습니다. 리스트를 바꾸는 메서드의 반환값과 바뀐 리스트를 따로 확인하세요.

컴프리헨션 `[n * n for n in numbers if n % 2 == 0]`은 다음 순서로 읽습니다.

1. `for n in numbers`: 항목을 하나씩 꺼냅니다.
2. `if n % 2 == 0`: 짝수만 통과시킵니다.
3. `n * n`: 통과한 항목을 제곱해 새 리스트에 넣습니다.

별칭 부분에서는 `alias`와 `original`의 `id`가 같은 객체를 가리킨다고 생각하면 됩니다. 그래서 `alias.append("C")`가 원본에서도 보입니다. 반면 `copied`는 별도 바깥 리스트여서 `"별도"`를 추가해도 원본은 바뀌지 않습니다. 함수에 리스트를 전달할 때 뜻하지 않게 원본을 바꾸는 실수가 자주 생기므로 입력 보존 조건을 항상 확인하세요.


### 짧은 확인 1 — 실행 전에 예측하세요 (약 1분)

다음 코드의 출력은 무엇일까요?

1. `values[1:4]`에 인덱스 4의 값이 포함될까요?
2. `b = a` 후 `b.append(30)`을 하면 `a`도 바뀔까요?
3. `len(values)`의 결과는 얼마일까요?

아래 셀을 실행해 확인합니다.

In [ ]:
values = [10, 20, 30, 40, 50]
a = [10, 20]
b = a
b.append(30)

print("values[1:4] =", values[1:4])
print("a =", a)
print("len(values) =", len(values))

assert values[1:4] == [20, 30, 40]
assert a == [10, 20, 30]
assert len(values) == 5
print("확인 완료!")

### 실습 1 — 기온 데이터 분석 함수 (약 5분)

`analyze_temperatures(values)`가 기온 리스트를 받아 다음 두 결과를 튜플로 반환하게 하세요.

1. 20도 이상인 값만 순서대로 담은 새 리스트
2. **전체 입력값** 의 산술 평균 원시값

입력 리스트는 비어 있지 않다고 가정하고, 함수 안에서 정렬·삭제·추가하지 마세요. 화면에는 평균을 소수 둘째 자리까지 `20.33`으로 표시하지만, 함수가 반환하는 원시 평균은 `20.333333...`이어야 합니다. 즉 `round`는 출력할 때만 사용하고 반환값을 미리 반올림하지 않습니다.

검증은 대표 입력 외에 값 하나인 입력을 다시 함수에 전달하고, 호출 전에 만든 `deepcopy`와 원본을 비교합니다.

**독립 입력·성공 예:** 입력 `[18,21,24,16,20,23]`이면 반환 튜플은 `([21,24,20,23], 122/6)`입니다. `warm`의 순서는 원본과 같고 평균은 따뜻한 날 네 개가 아닌 전체 여섯 값으로 계산합니다. 값 하나인 `[10]`에는 `([], 10.0)`을 반환합니다. 빈 리스트는 이번 계약에 포함되지 않습니다. 아래 TODO 함수만 수정하고 바로 뒤 학습자 검증을 실행하세요.


In [ ]:
def analyze_temperatures(values):
    """20도 이상 목록과 전체 평균 원시값을 반환합니다."""
    # TODO 1: values에서 20도 이상만 담은 새 리스트를 만드세요.
    warm = []
    # TODO 2: 전체 values의 합을 개수로 나눈 원시 평균을 구하세요.
    average = None
    return warm, average


temperatures = [18, 21, 24, 16, 20, 23]
warm_days, average_temperature = analyze_temperatures(temperatures)
display_average = (
    None if average_temperature is None else round(average_temperature, 2)
)

print('따뜻한 날:', warm_days)
print('평균 기온(화면 표시):', display_average)
print('평균 기온(원시값):', average_temperature)


In [ ]:
# 학습자 코드 실행 검증: 새 입력과 복사본을 호출 전에 준비합니다.
import copy

test_temperatures = [18, 21, 24, 16, 20, 23]
original_temperatures = copy.deepcopy(test_temperatures)
actual_warm_days, actual_average = analyze_temperatures(test_temperatures)
expected_average = sum(original_temperatures) / len(original_temperatures)

assert actual_warm_days == [21, 24, 20, 23]
assert abs(actual_average - expected_average) < 1e-12  # 원시 평균 검증
assert round(actual_average, 2) == 20.33               # 화면 표시값 설명과 일치
assert test_temperatures == original_temperatures      # 입력 보존

single_input = [20]
single_original = copy.deepcopy(single_input)
assert analyze_temperatures(single_input) == ([20], 20.0)
assert single_input == single_original
print('learner_verification=PASS')


In [ ]:
# 실습 1 정답: 리스트
def analyze_temperatures_solution(values):
    warm = [temperature for temperature in values if temperature >= 20]
    average = sum(values) / len(values)
    return warm, average


temperatures_answer = [18, 21, 24, 16, 20, 23]
warm_days_answer, average_temperature_answer = analyze_temperatures_solution(
    temperatures_answer
)
print('따뜻한 날:', warm_days_answer)
print('평균 기온(화면 표시):', round(average_temperature_answer, 2))
print('평균 기온(원시값):', average_temperature_answer)


### 정답 해설 — 입력부터 결과까지

**목록과 평균:** 조건 `temperature >= 20`으로 따뜻한 날을 새 목록에 모읍니다. 평균은 선택 목록이 아니라 전체 입력의 합을 전체 개수로 나눕니다. `[18,21,24,16,20,23]`에서는 선택값 `[21,24,20,23]`, 전체 합 122, 평균 122/6입니다.

계산 중에는 원래 실수값을 유지하고 사람이 보는 출력에서만 소수 둘째 자리로 반올림합니다. 새 목록을 만들어 입력을 보존합니다. 이 문제의 평균 입력은 비어 있지 않은 목록이라는 계약이므로 빈 목록 처리 여부를 다른 함수와 혼동하지 않습니다.

**확인 질문:** 20을 포함하는 이유와, 선택한 네 값으로 평균을 내면 질문이 달라지는 이유를 설명하세요.

**적용·재검증:** 자신의 TODO 셀에 필요한 로직을 반영하고 그 셀부터 실행한 뒤 학습자 검증을 실행합니다. `_answer`·`_solution`은 별도 참고 이름입니다. 정답 검증 PASS와 자신의 함수 PASS를 구분하고, 저장 셀은 자신의 검증 후 실행합니다.


In [ ]:
# 정답 코드 실행 검증: 원시 평균과 입력 보존을 따로 확인합니다.
import copy

solution_temperatures = [18, 21, 24, 16, 20, 23]
solution_original = copy.deepcopy(solution_temperatures)
solution_warm, solution_average = analyze_temperatures_solution(
    solution_temperatures
)
assert solution_warm == [21, 24, 20, 23]
assert abs(solution_average - sum(solution_original) / len(solution_original)) < 1e-12
assert round(solution_average, 2) == 20.33
assert solution_temperatures == solution_original
assert analyze_temperatures_solution([20]) == ([20], 20.0)
print('solution_verification=PASS')


## 2. 튜플(tuple): 순서가 있고 바꿀 수 없는 묶음

튜플은 보통 소괄호 `()`로 표현합니다. 리스트처럼 인덱싱과 슬라이싱이 가능하지만, 생성한 뒤 항목을 교체·추가·삭제할 수 없는 **불변(immutable)** 자료형입니다.

튜플이 잘 맞는 경우:

- `(x, y)` 좌표처럼 항목의 **개수와 의미가 고정** 된 데이터
- 함수에서 여러 값을 함께 반환할 때
- 딕셔너리 키처럼 **해시 가능한 값** 이 필요할 때

> 항목 하나짜리 튜플은 반드시 쉼표가 필요합니다: `(5,)`. `(5)`는 정수 5와 같습니다.

In [ ]:
section("튜플 생성과 접근")

rgb = (255, 128, 0)
single = ("Python",)
not_a_tuple = ("Python")

print("rgb:", rgb)
print("초록 성분:", rgb[1])
print("슬라이스:", rgb[:2])
print("single 타입:", type(single).__name__)
print("not_a_tuple 타입:", type(not_a_tuple).__name__)

# 아래 줄은 TypeError가 나므로 설명용으로 주석 처리했습니다.
# rgb[0] = 0

### 튜플 예제 출력 읽기

`rgb`는 빨강·초록·파랑 세 성분의 위치와 의미가 고정된 레코드입니다. `rgb[1]`은 위치 1, 즉 두 번째인 초록 성분 128입니다. 항목 하나인 `("Python",)`에는 쉼표가 있으므로 타입이 tuple이지만 `("Python")`은 괄호로 둘러싼 문자열일 뿐입니다.

불변이라는 말은 “읽을 수 없다”는 뜻이 아닙니다. 인덱싱과 순회는 가능하고, `rgb[0] = 0`처럼 기존 위치를 교체할 수 없다는 뜻입니다. 오류 예제의 주석을 잠시 해제하면 `TypeError`를 볼 수 있지만, 확인 후 다시 주석 처리해야 아래 셀을 계속 실행할 수 있습니다.


### 불변성에서 자주 생기는 오해

튜플의 불변성은 **튜플이 가리키는 항목을 다른 항목으로 교체할 수 없다** 는 뜻입니다. 튜플 안에 리스트 같은 가변 객체가 들어 있다면 그 **안쪽 객체 자체는 변경될 수 있습니다** .

또한 다음 두 언패킹을 구분하세요.

- 정확한 개수: `name, score = ("민지", 95)`
- 나머지 모으기: `first, *middle, last = [1, 2, 3, 4]`

별표로 받은 `middle`은 튜플이 아니라 **리스트** 입니다.

In [ ]:
section("튜플 언패킹")

student = ("민지", 95)
name, score = student
print(f"{name}의 점수는 {score}점입니다.")

first, *middle, last = (1, 2, 3, 4, 5)
print("first:", first)
print("middle:", middle, type(middle).__name__)
print("last:", last)

section("함수의 여러 반환값")
quotient, remainder = divmod(17, 5)
print("17 // 5 =", quotient)
print("17 % 5 =", remainder)

section("튜플 안 가변 객체")
course = ("Python", ["리스트", "튜플"])
course[1].append("딕셔너리")
print(course)

### 언패킹은 위치에 이름을 붙이는 일입니다

`name, score = student`는 오른쪽 튜플의 첫 항목을 `name`, 두 번째를 `score`에 연결합니다. 왼쪽 이름 수와 오른쪽 항목 수가 맞지 않으면 `ValueError`가 납니다. `*middle`은 남는 중간 항목을 리스트로 모아 개수가 달라도 받을 수 있게 합니다.

`divmod(17, 5)`는 몫과 나머지를 한 번에 튜플로 반환하고 곧바로 두 변수로 나눠 받습니다. 이후 OpenCV가 이미지 크기나 좌표를 여러 값으로 반환할 때 같은 언패킹을 사용합니다.

마지막 `course` 예제에서 튜플의 첫 위치와 둘째 위치는 바뀌지 않았지만, 둘째 위치가 가리키던 리스트 내부는 변경됩니다. **바깥 구조의 불변성과 안쪽 객체의 가변성은 별개** 입니다.


### 짧은 확인 2 + 실습 2 — 튜플과 이름 있는 `key` 함수 (약 5분)

먼저 `(42)`와 `(42,)` 중 어느 것이 튜플인지, `point`의 두 값을 어떻게 서로 바꿀지 생각해 보세요.

`max(records)`는 튜플의 첫 항목부터 비교합니다. 점수인 두 번째 항목을 기준으로 고르려면 다음처럼 **비교할 값을 반환하는 함수** 를 `key`에 전달합니다.

```python
def score_of(record):
    return record[1]

max(records, key=score_of)
```

`max`는 각 레코드를 `score_of`에 넣어 받은 점수를 비교하지만, 결과로는 점수만이 아니라 선택된 원래 튜플 전체를 반환합니다.

**TODO:** `point`를 언패킹해 `(3, 7)`을 만들고, `score_of`를 완성해 최고 점수 학생 튜플을 찾으세요. 입력 튜플과 레코드 목록은 바꾸지 않습니다. `lambda`는 같은 일을 짧게 적는 문법이지만 이 수업의 필수 범위에서는 이름 있는 함수를 먼저 사용합니다.

**입력·출력·완료 기준:** `point=(7,3)`에서 `x,y`를 읽고 `swapped=(y,x)` 순서의 새 튜플을 만듭니다. `score_of(record)`의 입력은 `(이름 문자열, 숫자 점수)`이고 반환은 두 번째 값 하나입니다. `records=[('민지',88),('준호',95),('서연',91)]`에 `max(records, key=score_of)`를 적용하면 `('준호',95)` 전체가 선택되어야 합니다. 아래 검증 셀이 `top_student`를 이 호출로 다시 계산하므로 직접 고정 정답을 대입하지 않습니다. `record[1]`에서 인덱스는 0부터 시작합니다.


In [ ]:
# TODO: 실습 2
point = (7, 3)
x, y = point
swapped = (None, None)  # TODO 1: x와 y로 (3, 7)을 만드세요.


def score_of(record):
    # TODO 2: (이름, 점수) 튜플에서 점수를 반환하세요.
    return None


records = [('민지', 88), ('준호', 95), ('서연', 91)]
top_student = None  # 검증 셀에서 score_of를 사용해 다시 계산합니다.

print('바꾼 좌표:', swapped)
print('최고 점수 학생:', top_student)


In [ ]:
# 학습자 코드 실행 검증: 함수를 완성한 뒤 결과를 다시 계산합니다.
top_student = max(records, key=score_of)
assert swapped == (3, 7)
assert score_of(('검증', 77)) == 77
assert top_student == ('준호', 95)
assert point == (7, 3)
assert records == [('민지', 88), ('준호', 95), ('서연', 91)]
print('learner_verification=PASS')


In [ ]:
# 실습 2 정답: 튜플
point_answer = (7, 3)
x_answer, y_answer = point_answer
swapped_answer = (y_answer, x_answer)


def score_of_solution(record):
    return record[1]


records_answer = [('민지', 88), ('준호', 95), ('서연', 91)]
top_student_answer = max(records_answer, key=score_of_solution)

print('바꾼 좌표:', swapped_answer)
print('최고 점수 학생:', top_student_answer)


### 정답 해설 — 입력부터 결과까지

**위치에 의미 붙이기:** `(7,3)`을 `x,y`로 풀고 `(y,x)`라는 새 튜플을 만들면 `(3,7)`입니다. 원래 튜플을 직접 수정한 것이 아닙니다.

학생 기록에서 `record[1]`은 점수입니다. `max(records, key=score_of_solution)`은 점수로 비교하되 반환값은 최고 점수 숫자만이 아니라 원래 기록 한 건입니다. 따라서 결과를 이름·점수로 다시 풀 수 있습니다.

**확인 질문:** `key` 없이 이름부터 비교하는 경우 왜 성적 순위와 달라질 수 있나요?

**적용·재검증:** 자신의 TODO 셀에 필요한 로직을 반영하고 그 셀부터 실행한 뒤 학습자 검증을 실행합니다. `_answer`·`_solution`은 별도 참고 이름입니다. 정답 검증 PASS와 자신의 함수 PASS를 구분하고, 저장 셀은 자신의 검증 후 실행합니다.


In [ ]:
# 정답 코드 실행 검증
assert swapped_answer == (3, 7)
assert score_of_solution(('검증', 77)) == 77
assert top_student_answer == ('준호', 95)
assert point_answer == (7, 3)
assert records_answer == [('민지', 88), ('준호', 95), ('서연', 91)]
print('solution_verification=PASS')


### 선택/자습 — 같은 `key`를 `lambda`로 짧게 쓰기

이름 있는 함수의 입력과 반환을 이해한 뒤에는 다음 두 표현이 같은 역할이라는 점만 확인하세요.

```python
def score_of(record):
    return record[1]

max(records, key=score_of)
max(records, key=lambda record: record[1])
```

`lambda record: record[1]`은 이름 없이 한 표현식의 결과를 반환하는 작은 함수입니다. 짧다고 항상 더 좋은 것은 아닙니다. 초심자 수업과 복잡한 비교 규칙에서는 이름 있는 함수가 의도를 설명하고 중간값을 시험하기 쉽습니다.


## 3. 딕셔너리(dict): 키와 값을 연결하는 구조

딕셔너리는 `{키: 값}` 쌍을 저장합니다. 정수 위치가 아니라 **고유한 키** 로 값을 찾습니다.

- 키는 중복될 수 없습니다. 같은 키에 다시 대입하면 값이 갱신됩니다.
- 키는 문자열, 숫자, 튜플처럼 **해시 가능한 불변 값** 이어야 합니다.
- 값에는 리스트나 다른 딕셔너리를 포함한 어떤 객체도 넣을 수 있습니다.
- 키 존재 확인은 `key in mapping`을 사용합니다.

In [ ]:
section("딕셔너리 생성·조회·갱신")

inventory = {
    "사과": 5,
    "바나나": 3,
    "귤": 8,
}

print("사과 재고:", inventory["사과"])

inventory["바나나"] = 4  # 갱신
inventory["포도"] = 2    # 새 키 추가
removed_stock = inventory.pop("귤")

print("갱신 후:", inventory)
print("삭제된 귤 재고:", removed_stock)
print("'포도'가 있나요?", "포도" in inventory)

### 딕셔너리 예제를 시간 순서로 읽기

초기 `inventory`는 상품 이름을 키, 재고 수를 값으로 연결합니다. `inventory["사과"]`는 위치를 세지 않고 이름으로 5를 찾습니다. 이미 있는 `"바나나"`에 대입하면 3이 4로 갱신되고, 없던 `"포도"`에 대입하면 새 항목이 생깁니다. `pop("귤")`은 항목을 지우면서 지운 값 8을 반환합니다.

중괄호는 집합에도 쓰이지만 `{키: 값}`처럼 콜론이 있으면 딕셔너리입니다. 빈 딕셔너리는 `{}`로 만듭니다. 키 철자와 대소문자가 다르면 별개의 키이므로 센서 이름을 일관된 형식으로 저장해야 합니다.

> **후속 과목 연결:** LLM/VLM의 JSON 응답, 물체의 `label`·`confidence`·`bbox`, ROS 설정은 키로 의미를 설명하는 딕셔너리 형태와 매우 비슷합니다.


### 안전한 조회와 흔한 함정

| 표현 | 키가 있을 때 | 키가 없을 때 | 용도 |
|---|---|---|---|
| `d[key]` | 값 반환 | `KeyError` | 반드시 있어야 하는 키 |
| `d.get(key)` | 값 반환 | `None` | 없어도 되는 키 |
| `d.get(key, default)` | 값 반환 | `default` | 기본값이 필요할 때 |
| `d.setdefault(key, default)` | 기존 값 반환 | 값을 추가한 뒤 반환 | 초기화와 조회를 함께 할 때 |

주의할 점:

- 값 자체가 `None`일 수 있다면 `get` 결과만으로 “키 없음”을 구분하지 말고 `key in d`를 사용합니다.
- 반복 중 딕셔너리의 크기를 바꾸면 오류가 날 수 있습니다. 삭제가 필요하면 `list(d)`처럼 키의 복사본을 순회합니다.
- `d.get("count", 0) + 1`은 간단한 빈도 계산에 유용합니다.

In [ ]:
section("딕셔너리 안전 조회")

profile = {"name": "서연", "nickname": None}

print("name:", profile["name"])
print("age 기본값:", profile.get("age", 0))
print("nickname 키 존재:", "nickname" in profile)
print("missing 키 존재:", "missing" in profile)

# 아래 줄은 KeyError가 나므로 설명용으로 주석 처리했습니다.
# print(profile["age"])

section("크기 변경이 필요할 때")
scores = {"민지": 95, "준호": 48, "서연": 82}
for student_name in list(scores):
    if scores[student_name] < 50:
        del scores[student_name]
print("필터링된 점수:", scores)

### “키 없음”과 “값이 None”을 구분하기

예제에서 `nickname` 키는 실제로 존재하지만 그 값이 `None`입니다. `missing` 키는 아예 없습니다. 두 키 모두 `profile.get(...)`만 보면 `None`이 나올 수 있으므로, 존재 여부 자체가 중요할 때는 `"nickname" in profile`처럼 검사합니다.

점수 삭제 예제에서 `list(scores)`는 현재 키의 복사본을 먼저 만듭니다. 그 복사본을 순회하므로 원본 딕셔너리에서 항목을 삭제해도 순회 대상의 크기는 바뀌지 않습니다. 반복 중인 원본의 크기를 직접 바꾸면 `RuntimeError`가 날 수 있습니다.


### 순회와 딕셔너리 컴프리헨션

- `for key in d`: 키 순회
- `for value in d.values()`: 값 순회
- `for key, value in d.items()`: 키와 값 함께 순회
- `{key_expr: value_expr for item in iterable}`: 새 딕셔너리 생성

키로 리스트는 사용할 수 없지만, 모든 항목이 해시 가능하다면 튜플은 키로 사용할 수 있습니다.

In [ ]:
section("딕셔너리 순회")

prices = {"연필": 500, "노트": 1500, "지우개": 700}

for product, price in prices.items():
    print(f"{product}: {price:,}원")

discounted = {
    product: int(price * 0.9)
    for product, price in prices.items()
    if price >= 700
}
print("10% 할인 대상:", discounted)

section("튜플을 키로 사용")
seats = {
    (1, "A"): "예약",
    (1, "B"): "비어 있음",
}
print("1열 A:", seats[(1, "A")])

### 딕셔너리 순회와 결과 읽기

`prices.items()`는 `(상품명, 가격)` 쌍을 하나씩 주고, 두 변수로 언패킹합니다. 할인 딕셔너리 컴프리헨션은 가격이 700원 이상인 항목만 골라 가격에 0.9를 곱한 새 딕셔너리를 만듭니다. 원본 `prices`는 바뀌지 않습니다.

`seats[(1, "A")]`에서 키는 두 값으로 된 튜플 하나입니다. `seats[1, "A"]`도 Python에서는 같은 튜플 키를 뜻합니다. 좌표나 격자 위치처럼 두 값의 조합이 하나의 고유한 이름표일 때 유용합니다.


### 짧은 확인 3 + 실습 3 (약 5분)

먼저 없는 키를 `d[key]`로 읽을 때의 예외와 `d.get(key, 0)`의 차이를 떠올려 보세요.

**실습 목적:** 딕셔너리의 기본값 조회로 빈도를 누적하고, 컴프리헨션으로 새 딕셔너리를 필터링합니다.

**실습 내용:** `dict.get` 빈도 누적과 딕셔너리 컴프리헨션 필터링을 실습합니다.

**구현 항목**

1. `words`를 순회하며 `frequencies[word] = frequencies.get(word, 0) + 1` 방식으로 횟수를 셉니다.
2. 빈도가 2 이상인 키·값만 `frequent`에 담는 딕셔너리 컴프리헨션을 작성합니다.

**입력·출력/제약:** 입력은 문자열 목록이고 출력은 두 딕셔너리입니다. `collections.Counter` 대신 반복문과 `dict.get`을 사용하며, 원본 단어 목록은 변경하지 않습니다.

**완료 기준:** `frequencies == {"python": 3, "data": 2, "queue": 1}`이고 `frequent == {"python": 3, "data": 2}`이면 완료입니다.



In [ ]:
# TODO: 실습 3
words = ["python", "data", "python", "queue", "data", "python"]

frequencies = {}
# TODO: words를 순회하며 frequencies를 갱신하세요.

frequent = {}  # TODO: 빈도가 2 이상인 키-값만 딕셔너리 컴프리헨션으로 만들기

print("빈도표:", frequencies)
print("2회 이상:", frequent)

In [ ]:
# 학습자 코드 실행 검증
assert frequencies == {'python': 3, 'data': 2, 'queue': 1}
assert frequent == {'python': 3, 'data': 2}
print('learner_verification=PASS')

In [ ]:
# 실습 3 정답: 딕셔너리
words_answer = ["python", "data", "python", "queue", "data", "python"]

frequencies_answer = {}
for word in words_answer:
    # 처음 보는 단어는 기본값 0에서, 이미 본 단어는 기존 횟수에서 1 증가한다.
    frequencies_answer[word] = frequencies_answer.get(word, 0) + 1

# 빈도표를 순회하며 기준을 만족하는 항목만 새 딕셔너리에 복사한다.
frequent_answer = {
    word: count
    for word, count in frequencies_answer.items()
    if count >= 2
}

print("빈도표:", frequencies_answer)
print("2회 이상:", frequent_answer)

assert frequencies_answer == {"python": 3, "data": 2, "queue": 1}
assert frequent_answer == {"python": 3, "data": 2}

### 정답 해설 — 입력부터 결과까지

**빈도 누적:** `get(word,0)`으로 이전 횟수를 읽고 1을 더합니다. 처음 본 단어는 0에서, 이미 본 단어는 누적값에서 시작합니다. python→data→python 순서라면 python 횟수는 1→1→2입니다.

최종 빈도는 python 3, data 2, queue 1입니다. 그 뒤 빈도가 2 이상인 항목을 새 딕셔너리에 골라 python·data만 남깁니다. 누적과 필터를 나누면 원래 빈도표를 다시 확인할 수 있습니다.

**확인 질문:** 매번 `frequencies[word] = 1`로 쓰면 무엇을 잃나요?

**적용·재검증:** 자신의 TODO 셀에 필요한 로직을 반영하고 그 셀부터 실행한 뒤 학습자 검증을 실행합니다. `_answer`·`_solution`은 별도 참고 이름입니다. 정답 검증 PASS와 자신의 함수 PASS를 구분하고, 저장 셀은 자신의 검증 후 실행합니다.


In [ ]:
# 정답 코드 실행 검증
assert frequencies_answer == {'python': 3, 'data': 2, 'queue': 1}
assert frequent_answer == {'python': 3, 'data': 2}
print('solution_verification=PASS')

## 수업 핵심 — 제조 센서 보고서 (약 16분)

입력은 `(설비명, 온도)` 튜플의 리스트입니다. 결과는 설비 이름으로 `count`, `mean`, `alarm`을 찾는 중첩 딕셔너리입니다. `alarm`은 평균이 아니라 해당 설비에서 **한 번이라도 30.0도 이상** 이 측정되면 `True`입니다.

한 함수에서 두 일을 동시에 해결하지 않고 다음 두 단계로 나눕니다.

1. `group_sensor_readings`: 같은 설비의 온도를 리스트로 모읍니다.
2. `summarize_sensor_groups`: 모은 리스트에서 개수·평균·경보를 계산합니다.

두 함수 모두 입력을 수정하지 않아야 하며 빈 입력은 빈 딕셔너리로 처리합니다. 실행 전 예측: 설비 A의 평균이 30도보다 낮아도 경보가 켜질 수 있을까요?


### 문제를 코드로 옮기기 전에 입·출력을 그려 보세요

<!-- procedure:lesson-02-ef4c55:start -->
![센서 기록에서 보고서까지](../교재그림/절차/lesson-02.svg)

원본 → 그룹 → 요약을 각각 확인합니다. 그룹 오류는 통계 함수에서 숨기지 않습니다.
<!-- procedure:lesson-02-ef4c55:end -->


입력은 시간 순서의 측정 목록입니다.

```text
[("A", 20.0), ("B", 25.0), ("A", 30.0)]
       │                 │
       └──── 같은 A ─────┘
```

함수의 반환 결과는 설비 이름으로 요약을 찾는 중첩 딕셔너리입니다.

```text
{
  "A": {"count": 2, "mean": 25.0, "alarm": True},
  "B": {"count": 1, "mean": 25.0, "alarm": False}
}
```

- 바깥 키 `"A"`, `"B"`: 설비 이름
- `count`: 해당 설비의 측정 개수
- `mean`: 해당 설비 값의 합 ÷ 개수
- `alarm`: 평균과 무관하게 측정 중 하나라도 30.0 이상인지 여부

설비 A의 평균은 `(20.0 + 30.0) / 2 == 25.0`이지만 30.0인 측정이 하나 있으므로 경보는 참입니다. 평균 조건과 최대값 조건을 혼동하지 마세요.

### 단계별 구현 전략

1. 빈 그룹 딕셔너리를 준비합니다.
2. 각 `(설비명, 온도)` 튜플을 순회하며 같은 설비의 온도를 한 리스트에 모읍니다.
3. 그룹 딕셔너리를 직접 출력해 `{"A": [20.0, 30.0], "B": [25.0]}` 모양인지 확인합니다.
4. 각 설비의 값 리스트에서 개수, 평균, 경보를 계산해 결과 딕셔너리를 만듭니다.
5. 빈 입력에서는 순회가 한 번도 일어나지 않아 빈 딕셔너리가 반환되는지 확인합니다.
6. 원본 `rows`를 정렬하거나 삭제하지 않았는지 확인합니다.

함수 전체가 어렵다면 먼저 그룹 만들기까지만 작성하고 `print(groups)`로 확인한 뒤 요약 단계로 넘어가세요. 검증 셀의 여러 `assert`는 대표 입력, 빈 입력, 경계값 30.0, 음수, 입력 보존을 각각 확인합니다.

### 흔한 실수와 해결

| 증상 | 가능한 원인 | 확인 방법 |
|---|---|---|
| A의 count가 1 | 기존 리스트 대신 매번 새 리스트를 넣음 | 각 반복 뒤 `groups` 출력 |
| A의 mean이 50 | 합계를 개수로 나누지 않음 | `sum(values)`와 `len(values)` 따로 출력 |
| alarm이 False | 평균에 30 기준을 적용함 | 요구사항의 “한 번이라도”를 다시 읽기 |
| 빈 입력에서 오류 | 값이 없는 상태에서 `max` 실행 | 결과 생성이 실제 그룹만 순회하는지 확인 |
| 검증이 계속 이전 결과를 봄 | TODO 셀 수정 후 실행하지 않음 | TODO 셀 실행 번호가 최신인지 확인 |

이 실습은 이후 카메라 프레임마다 나온 검출 결과를 물체 종류별로 묶고 통계를 만들거나, ROS 메시지를 센서별 상태로 요약하는 처리의 작은 모형입니다.


### 1단계 TODO — 설비별 온도 그룹 만들기 (약 7분)

`rows`의 각 튜플을 `name, value`로 언패킹하고, `groups[name]` 리스트에 온도를 순서대로 추가하세요. 처음 만난 설비에는 빈 리스트가 필요합니다.

힌트: `groups.setdefault(name, []).append(value)`를 사용할 수 있습니다.

현재 입력의 중간 결과는 `{"A": [20.0, 30.0], "B": [25.0]}` 모양이어야 합니다. 통계는 아직 계산하지 않습니다.

**입력·반환·경계:** `group_sensor_readings(rows)`의 `rows`는 `(설비 이름 문자열, 온도 숫자)` 튜플들의 리스트이며, 반환은 설비 이름→온도 리스트인 새 딕셔너리입니다. 빈 입력 `[]`는 `{}`, `[('C',-5.0)]`는 `{'C':[-5.0]}`를 반환합니다. 원본 목록·튜플을 수정하지 않고 같은 설비의 원소 순서를 보존합니다. 한 설비를 다시 만나도 이전 값을 지우지 않아야 합니다. 바로 뒤 검증이 대표 입력·빈 입력·음수·원본 보존을 확인합니다.


In [ ]:
readings = [("A", 20.0), ("B", 25.0), ("A", 30.0)]


def group_sensor_readings(rows):
    """설비 이름별 온도 리스트를 반환합니다."""
    groups = {}
    # TODO: rows를 순회하며 같은 설비의 값을 groups에 모으세요.
    return groups


groups = group_sensor_readings(readings)
print('1단계 그룹:', groups)


In [ ]:
# 1단계 학습자 코드 실행 검증: 호출 전에 새 입력과 복사본을 만듭니다.
import copy

stage1_rows = [("A", 20.0), ("B", 25.0), ("A", 30.0)]
stage1_original = copy.deepcopy(stage1_rows)
stage1_groups = group_sensor_readings(stage1_rows)

assert stage1_groups == {"A": [20.0, 30.0], "B": [25.0]}
assert group_sensor_readings([]) == {}
assert group_sensor_readings([("C", -5.0)]) == {"C": [-5.0]}
assert stage1_rows == stage1_original
print('learner_verification=PASS')


In [ ]:
def group_sensor_readings_solution(rows):
    groups = {}
    for name, value in rows:
        groups.setdefault(name, []).append(value)
    return groups


### 정답 해설 — 입력부터 결과까지

**그룹 만들기:** `groups`는 설비 이름을 key로, 측정값 목록을 value로 둡니다. `setdefault(name,[])`는 처음 본 설비에만 새 목록을 만들고, 이미 있는 설비는 그 목록을 돌려줍니다. 이어지는 `append(value)`가 현재 측정값을 추가합니다.

`[("A",20),("B",25),("A",30)]`을 읽으면 A:[20] → A:[20],B:[25] → A:[20,30],B:[25]가 됩니다. 원본 행은 수정하지 않습니다. 빈 입력은 반복이 한 번도 돌지 않으므로 `{}`입니다.

**확인 질문:** 매 행마다 `groups[name]=[value]`로 대입하면 A의 어떤 측정이 사라지나요?

**적용·재검증:** 자신의 TODO 셀에 필요한 로직을 반영하고 그 셀부터 실행한 뒤 학습자 검증을 실행합니다. `_answer`·`_solution`은 별도 참고 이름입니다. 정답 검증 PASS와 자신의 함수 PASS를 구분하고, 저장 셀은 자신의 검증 후 실행합니다.


In [ ]:
# 1단계 정답 코드 실행 검증
import copy

solution_rows = [("A", 20.0), ("B", 25.0), ("A", 30.0)]
solution_rows_original = copy.deepcopy(solution_rows)
assert group_sensor_readings_solution(solution_rows) == {
    "A": [20.0, 30.0], "B": [25.0]
}
assert solution_rows == solution_rows_original
assert group_sensor_readings_solution([]) == {}
print('solution_verification=PASS')


### 2단계 TODO — 그룹을 통계로 요약하기 (약 9분)

`groups.items()`를 순회해 설비마다 다음 세 값을 계산하세요.

- `count`: `len(values)`
- `mean`: `sum(values) / len(values)`
- `alarm`: `max(values) >= 30.0`

1단계가 통과한 뒤 시작하세요. `groups`가 비어 있으면 결과도 빈 딕셔너리여야 하며 입력 딕셔너리와 내부 리스트를 수정하지 않습니다.

**이 TODO를 따로 읽을 때의 입력과 출력:** `summarize_sensor_groups(groups)`는 `{'A':[20.0,30.0], 'B':[25.0]}`를 받아 `{'A':{'count':2,'mean':25.0,'alarm':True}, 'B':{'count':1,'mean':25.0,'alarm':False}}`를 반환합니다. `count`는 정수, `mean`은 전체 값의 평균, `alarm`은 bool입니다. 각 `values`는 비어 있지 않은 숫자 리스트라고 가정합니다. 빈 그룹 `{}`은 허용하지만 `{'A':[]}`는 이번 입력 계약에 없습니다.

**1단계 위치와 필요한 내용:** 바로 위 「1단계 TODO — 설비별 온도 그룹 만들기」의 `group_sensor_readings`가 `[(설비명,온도), ...]`를 순회해 설비마다 리스트를 만드는 함수입니다. 이 단계 검증은 자체 입력을 준비하므로 2단계 함수만으로도 확인할 수 있고, 최종 JSON 저장은 두 함수 모두 필요합니다. 한 값 `30.0`에서도 `alarm=True`여야 하며 평균을 기준과 비교하지 마세요.


In [ ]:
def summarize_sensor_groups(groups):
    """설비별 온도 리스트에서 통계 딕셔너리를 반환합니다."""
    summary = {}
    # TODO: 각 설비의 count, mean, alarm을 summary에 저장하세요.
    return summary


report = summarize_sensor_groups(groups)
print('2단계 보고서:', report)


In [ ]:
# 2단계 학습자 코드 실행 검증: 호출 전에 새 입력과 복사본을 만듭니다.
import copy

stage2_groups = {"A": [20.0, 30.0], "B": [25.0]}
stage2_original = copy.deepcopy(stage2_groups)
stage2_report = summarize_sensor_groups(stage2_groups)

assert stage2_report == {
    "A": {"count": 2, "mean": 25.0, "alarm": True},
    "B": {"count": 1, "mean": 25.0, "alarm": False},
}
assert summarize_sensor_groups({}) == {}
assert summarize_sensor_groups({"C": [30.0]})["C"]["alarm"] is True
assert summarize_sensor_groups({"C": [-5.0]})["C"]["mean"] == -5.0
assert stage2_groups == stage2_original
print('learner_verification=PASS')


In [ ]:
def summarize_sensor_groups_solution(groups):
    summary = {}
    for name, values in groups.items():
        summary[name] = {
            "count": len(values),
            "mean": sum(values) / len(values),
            "alarm": max(values) >= 30.0,
        }
    return summary


### 정답 해설 — 입력부터 결과까지

**통계로 바꾸기:** `groups.items()`에서 설비 이름과 값 목록을 함께 읽습니다. `len(values)`는 개수, `sum(values)/len(values)`는 평균, `max(values) >= 30.0`은 최댓값이 경보 기준에 도달했는지입니다. 비어 있지 않은 숫자 목록에서는 `any(value >= 30.0 for value in values)`와 같은 뜻입니다. 새 `summary`를 만들므로 그룹 원본을 보존합니다.

A:[20,30]은 count=2, mean=25.0, alarm=True입니다. 평균 25가 낮아도 30이라는 경계값이 있으므로 경보입니다. 빈 그룹 사전은 `{}`를 반환합니다. 그룹 값의 빈 목록은 앞의 그룹 생성 함수가 만들지 않는 입력이므로 그 계약도 함께 읽습니다.

**확인 질문:** A:[25,25]는 평균이 같은데 왜 경보가 다른가요?

**적용·재검증:** 자신의 TODO 셀에 필요한 로직을 반영하고 그 셀부터 실행한 뒤 학습자 검증을 실행합니다. `_answer`·`_solution`은 별도 참고 이름입니다. 정답 검증 PASS와 자신의 함수 PASS를 구분하고, 저장 셀은 자신의 검증 후 실행합니다.


In [ ]:
# 2단계 정답 코드 실행 검증
import copy

solution_groups = {"A": [20.0, 30.0], "B": [25.0]}
solution_groups_original = copy.deepcopy(solution_groups)
assert summarize_sensor_groups_solution(solution_groups) == {
    "A": {"count": 2, "mean": 25.0, "alarm": True},
    "B": {"count": 1, "mean": 25.0, "alarm": False},
}
assert solution_groups == solution_groups_original
assert summarize_sensor_groups_solution({}) == {}
print('solution_verification=PASS')


### 자신의 결과 저장

두 단계의 학습자 검증을 통과한 후 자신의 두 함수를 연결해 JSON으로 저장합니다. Data는 호스트에 연결되어 컨테이너 삭제 후에도 남습니다.


### JSON 파일을 저장하는 이유와 결과 확인법

메모리의 변수는 커널이나 컨테이너가 끝나면 사라집니다. JSON은 딕셔너리·리스트·문자열·숫자·불리언을 사람이 읽을 수 있는 텍스트로 저장하는 널리 쓰이는 형식입니다.

저장 셀의 흐름은 다음과 같습니다.

1. 자신의 `summarize_sensors`를 다시 호출해 `report`를 만듭니다.
2. 학습자 검증에서 확인한 현재 입력의 기대 구조와 같은지 다시 확인합니다.
3. `json.dumps(..., ensure_ascii=False, indent=2)`로 들여쓴 JSON 문자열을 만듭니다.
4. `write_text`로 `Data` 아래 파일에 씁니다.
5. 파일을 다시 읽어 원래 딕셔너리와 같은지 확인합니다.

출력된 경로를 JupyterLab 파일 브라우저에서 열어 `A`, `B`, `count`, `mean`, `alarm`을 직접 확인하세요. JSON의 `true`/`false`는 Python에서 읽으면 `True`/`False`가 됩니다. 컨테이너를 `--rm`으로 삭제해도 호스트와 연결한 `Data`에 있는 이 파일은 남습니다.


In [ ]:
import json


def summarize_sensors(rows):
    grouped = group_sensor_readings(rows)
    return summarize_sensor_groups(grouped)


report = summarize_sensors(readings)
assert report == {
    "A": {"count": 2, "mean": 25.0, "alarm": True},
    "B": {"count": 1, "mean": 25.0, "alarm": False},
}
result_path = OUTPUT / "sensor_report.json"
result_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
assert json.loads(result_path.read_text(encoding="utf-8")) == report
print(result_path)


## 수업 핵심 완료 점검

- 리스트·튜플·딕셔너리의 수업 핵심 TODO를 검증했나요?
- 센서의 그룹 만들기와 요약 만들기 검증을 각각 통과했나요?
- 평균과 경보 조건이 다른 이유를 설명할 수 있나요?
- `Data/outputs/02.Python/sensor_report.json`을 열어 자신의 결과를 확인하세요.
- 선택/자습은 이어지는 큐·주문 처리 절을 진행하고, 마지막 전체 마무리로 이동하세요.


### 후속 과목에서 자료구조를 보는 관점

| 이후 과목 | 자주 보게 될 데이터 형태 | 오늘의 연결점 |
|---|---|---|
| LLM/VLM | 메시지 목록과 JSON 응답 | 리스트 안 딕셔너리, 키로 값 조회 |
| OpenCV | 이미지 배열, 좌표와 색상 | 순서·인덱스, 고정 좌표 튜플 |
| MediaPipe | 여러 랜드마크와 각 좌표 | 리스트 순회, 중첩 구조 |
| ROS | 메시지 필드와 수신 대기열 | 속성별 값, FIFO 처리 |
| VLA | 관측·판단·명령 기록 | 중첩 딕셔너리와 결과 목록 |

새 라이브러리의 출력이 복잡해 보이면 먼저 `type(value)`, `len(value)`, `value.keys()`를 차례로 확인하세요. 전체를 한 번에 이해하기보다 바깥 구조가 리스트인지 딕셔너리인지 확인하고, 항목 하나를 꺼내 다시 타입과 키를 살펴보면 됩니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 설비별 그룹·통계 요약과 JSON 저장 확인; 큐·서점 자습 제외

[1일차 슬라이드 54쪽](https://docs.google.com/presentation/d/1pQvfu3XzIfFL9FFvJkP8a6iNKJMjQFGOQIboEF6DRjg/edit#slide=id.d1_s049) · [배포 PDF 54쪽](../슬라이드/1일차.pdf#page=54)로 돌아가 **54–65쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **선택/자습 확장:** 수업 핵심 완료 후 큐→주문 처리 순서로 진행하세요.

## 4. 큐(queue): 먼저 들어온 항목을 먼저 처리하기

큐는 **FIFO(First In, First Out)** 규칙을 따릅니다. 줄을 먼저 선 사람이 먼저 서비스를 받는 것과 같습니다.

Python 표준 라이브러리의 `collections.deque`는 양쪽 끝에서 빠르게 추가·삭제하도록 설계되었습니다.

- 뒤에 넣기(enqueue): `queue.append(item)`
- 앞에서 꺼내기(dequeue): `queue.popleft()`
- 앞 항목 보기(peek): `queue[0]` (비어 있지 않을 때)
- 비었는지 확인: `if queue:`


In [ ]:
from collections import deque

section("deque로 FIFO 큐 만들기")

waiting = deque(["민지", "준호"])
waiting.append("서연")
print("대기열:", list(waiting))
print("다음 고객:", waiting[0])

served = waiting.popleft()
print("서비스 완료:", served)
print("남은 대기열:", list(waiting))

while waiting:
    print("처리:", waiting.popleft())

print("비어 있나요?", not waiting)

### 큐 구현 선택과 성능

| 구현 | 뒤에 추가 | 앞에서 제거 | 스레드 안전 | 추천 상황 |
|---|---:|---:|---|---|
| 리스트 `append / pop(0)` | 평균 O(1) | **O(n)** | 보장 안 됨 | 아주 작은 임시 데이터 외에는 큐로 비추천 |
| `collections.deque` | O(1) | **O(1)** | 복합 작업 전체의 동기화는 별도 고려 | 일반적인 단일 스레드 FIFO |
| `queue.Queue` | 내부 동기화 | 내부 동기화 | **예** | 여러 스레드 사이의 작업 전달 |

리스트에서 `pop(0)`을 하면 뒤의 모든 항목을 한 칸씩 옮겨야 합니다. 큐에는 `deque.popleft()`가 더 효율적입니다.

### 핵심 이론 · 시간 복잡도와 저장 구조

리스트는 객체 참조를 연속된 영역에 두기 때문에 앞 항목을 지우면 뒤의 약 `n-1`개 참조를 당겨야 해 **O(n)** 입니다. `deque`는 여러 블록을 양방향으로 연결해 양 끝의 추가·삭제를 **O(1)** 로 유지합니다. 딕셔너리의 키 조회는 해시 테이블 덕분에 **평균 O(1)** 이지만, 충돌이나 크기 조정이 일어나는 특정 순간에는 더 오래 걸릴 수 있습니다. 따라서 평균·분할상환 복잡도는 개별 실행 시간의 보장이 아니라, 입력 크기가 커질 때 비용이 어떻게 증가하는지를 설명하는 모델입니다. 자주 수행할 연산을 먼저 정한 뒤 구조를 선택하세요.

### 흔한 함정

- 빈 `deque`에서 `popleft()`를 호출하면 `IndexError`가 납니다. 먼저 `if queue:`로 확인하세요.
- 변수 이름을 `list`, `dict`, `tuple`, `queue` 같은 내장 이름으로 짓지 않는 편이 좋습니다.
- `deque`의 개별 연산이 안전하게 구현되어 있어도, “비었는지 확인한 뒤 꺼내기” 같은 **여러 연산의 조합** 은 멀티스레드 경쟁 상태가 생길 수 있습니다. 그런 작업 전달에는 `queue.Queue`를 사용합니다.

In [ ]:
from queue import Empty, Queue

section("멀티스레드 작업 전달용 queue.Queue")

task_queue = Queue()
task_queue.put("파일 읽기")
task_queue.put("결과 저장")

while True:
    try:
        task = task_queue.get_nowait()
    except Empty:
        break
    else:
        print("작업 처리:", task)
        task_queue.task_done()

print("남은 작업 수:", task_queue.qsize())

# 이 예제는 한 스레드에서만 실행합니다. Queue는 여러 스레드의 put/get을 조율합니다.
# 여러 소비자는 qsize로 성공을 예측하지 않고 get/get_nowait의 결과와 Empty를 처리합니다.
# 실제 스레드 생성·종료와 대기 조율은 이 배포 자료의 범위 밖입니다.

### 짧은 확인 4 + 실습 4 (약 4분)

먼저 FIFO의 앞 항목을 꺼내는 메서드와 리스트 `pop(0)`보다 적합한 이유를 떠올려 보세요.

**실습 목적:** `deque`를 FIFO 대기열로 사용해 도착 순서를 보존하며 모든 항목을 처리합니다.

**실습 내용:** `while`과 `deque.popleft()`를 이용한 FIFO 소비를 실습합니다.

**구현 항목:** `waiting_line`이 빌 때까지 `while`로 반복하고, 매번 `popleft()`로 꺼낸 이름을 `served_customers.append(...)`로 기록하세요.

**입력·출력/제약:** 입력은 이름 세 개가 든 `deque`이며 출력은 처리 순서 리스트입니다. 인덱싱이나 `pop()`으로 뒤에서 꺼내지 말고, 처리 후 입력 큐를 비워야 합니다. 이 실습은 단일 스레드 상황입니다.

**완료 기준:** `served_customers == ["민지", "준호", "서연"]`이고 `not waiting_line`이 참이면 완료입니다.



In [ ]:
# TODO: 실습 4
from collections import deque

waiting_line = deque(["민지", "준호", "서연"])
served_customers = []

# TODO: while과 popleft를 사용해 모두 처리하세요.

print("처리 순서:", served_customers)
print("대기열이 비었나요?", not waiting_line)

In [ ]:
# 학습자 코드 실행 검증
assert served_customers == ['민지', '준호', '서연']
assert not waiting_line
print('learner_verification=PASS')

In [ ]:
# 실습 4 정답: 큐
from collections import deque

waiting_line_answer = deque(["민지", "준호", "서연"])
served_customers_answer = []

while waiting_line_answer:  # 빈 큐에서 popleft하지 않도록 항목이 있을 때만 반복
    # 왼쪽의 가장 오래 기다린 고객을 꺼내 처리 순서에 기록한다.
    served_customers_answer.append(waiting_line_answer.popleft())

print("처리 순서:", served_customers_answer)

assert served_customers_answer == ["민지", "준호", "서연"]
assert not waiting_line_answer

### 정답 해설 — 입력부터 결과까지

**FIFO 처리:** `deque`의 왼쪽에서 `popleft()`로 꺼내므로 민지→준호→서연 순서를 지킵니다. `while waiting_line`은 큐가 비었을 때 종료되어 빈 큐에서 꺼내는 오류를 막습니다.

대기 큐는 줄어들고 처리 결과 목록은 늘어납니다. 여기서는 큐를 소비하는 것이 과제의 의도이며, 입력 보존을 요구한 앞의 통계 함수와 다릅니다.

**확인 질문:** 오른쪽 `pop()`을 쓰면 먼저 온 사람이 먼저 처리되나요? 이 절은 선택/자습입니다.

**적용·재검증:** 자신의 TODO 셀에 필요한 로직을 반영하고 그 셀부터 실행한 뒤 학습자 검증을 실행합니다. `_answer`·`_solution`은 별도 참고 이름입니다. 정답 검증 PASS와 자신의 함수 PASS를 구분하고, 저장 셀은 자신의 검증 후 실행합니다.


In [ ]:
# 정답 코드 실행 검증
assert served_customers_answer == ['민지', '준호', '서연']
assert not waiting_line_answer
print('solution_verification=PASS')

> **선택/자습 확장:** 수업 핵심 완료 후 큐→주문 처리 순서로 진행하세요.

## 5. 통합 도전 — 서점 주문 처리기 (약 8분)

**실습 목적:** 딕셔너리 재고, 튜플 주문, FIFO 큐, 결과 리스트를 한 처리 흐름에 결합합니다.

**실습 내용:** 네 자료구조의 역할을 나누어 주문 성공·실패 상태 전이를 실습합니다.

**구현 항목**

1. `orders`가 빌 때까지 주문 튜플을 `popleft()`하고 세 값으로 언패킹합니다.
2. 상품이 없으면 `(주문번호, "없는 상품")`, 수량이 부족하면 `(주문번호, "재고 부족")`을 `rejected`에 추가합니다.
3. 성공하면 재고를 차감하고 `{"order_id": 번호, "item": 상품, "quantity": 수량}`을 `processed`에 추가합니다.

**입력·출력/제약:** `stock`은 상품별 비음수 재고 딕셔너리, `orders`는 `(주문번호, 상품명, 양의 수량)` 튜플의 `deque`입니다. FIFO 순서와 각 결과 레코드의 명세를 보존하고, 실패 주문은 재고를 바꾸지 않아야 합니다. 모든 주문을 정확히 한 번 처리하세요.

**완료 기준:** 성공 주문 번호가 `A-101`, `A-103`, 실패 결과가 `A-102` 재고 부족과 `A-104` 없는 상품, 최종 재고가 `{"노트": 3, "연필": 3}`이고 `orders`가 비어 있으면 완료입니다.



In [ ]:
# TODO: 통합 도전
from collections import deque

stock = {"노트": 5, "연필": 10}
orders = deque([
    ("A-101", "노트", 2),
    ("A-102", "연필", 12),
    ("A-103", "연필", 7),
    ("A-104", "지우개", 1),
])

processed = []
rejected = []

# TODO:
# while orders:
#     주문 튜플을 popleft로 꺼내 언패킹합니다.
#     상품 존재 여부와 재고를 확인합니다.
#     성공 또는 실패 결과를 알맞은 리스트에 추가합니다.

print("최종 재고:", stock)
print("성공:", processed)
print("실패:", rejected)
print("남은 주문:", list(orders))

In [ ]:
# 학습자 코드 실행 검증
assert stock == {'노트': 3, '연필': 3}
assert [row['order_id'] for row in processed] == ['A-101', 'A-103']
assert rejected == [('A-102', '재고 부족'), ('A-104', '없는 상품')]
assert not orders
print('learner_verification=PASS')

In [ ]:
# 통합 도전 정답
from collections import deque

stock_answer = {"노트": 5, "연필": 10}
orders_answer = deque([
    ("A-101", "노트", 2),
    ("A-102", "연필", 12),
    ("A-103", "연필", 7),
    ("A-104", "지우개", 1),
])

processed_answer = []
rejected_answer = []

while orders_answer:
    # 가장 먼저 들어온 주문을 한 번 꺼내 고정 형식의 세 필드로 언패킹한다.
    order_id, item, quantity = orders_answer.popleft()

    if item not in stock_answer:
        rejected_answer.append((order_id, "없는 상품"))  # 존재 여부를 먼저 검사해 안전하게 조회
    elif stock_answer[item] < quantity:
        rejected_answer.append((order_id, "재고 부족"))  # 실패 주문은 재고를 변경하지 않음
    else:
        stock_answer[item] -= quantity  # 성공한 경우에만 재고 차감
        processed_answer.append({
            "order_id": order_id,
            "item": item,
            "quantity": quantity,
        })  # 명세된 딕셔너리 형태로 성공 결과 기록

print("최종 재고:", stock_answer)
print("성공:", processed_answer)
print("실패:", rejected_answer)

assert stock_answer == {"노트": 3, "연필": 3}
assert [row["order_id"] for row in processed_answer] == ["A-101", "A-103"]
assert rejected_answer == [
    ("A-102", "재고 부족"),
    ("A-104", "없는 상품"),
]
assert not orders_answer
print("통합 도전 검증 완료!")

### 정답 해설 — 입력부터 결과까지

**주문마다 검사 후 갱신:** 큐에서 주문을 꺼내고 상품 존재·주문 수량·재고를 검사합니다. 거절한 주문은 거절 목록에 이유를 남기며 재고를 줄이지 않습니다. 승인한 주문만 재고와 처리 목록을 함께 갱신합니다.

연필 재고 10에서 12개 요청을 거절하면 여전히 10개입니다. 다음 7개 요청은 승인할 수 있고 3개가 남습니다. 검증 전에 재고를 먼저 빼면 거절 이후 주문에도 오류가 전파됩니다.

**확인 질문:** 실패한 주문이 재고를 바꾸지 않는다는 것을 어느 검사로 확인하나요? 이 절은 선택/자습입니다.

**적용·재검증:** 자신의 TODO 셀에 필요한 로직을 반영하고 그 셀부터 실행한 뒤 학습자 검증을 실행합니다. `_answer`·`_solution`은 별도 참고 이름입니다. 정답 검증 PASS와 자신의 함수 PASS를 구분하고, 저장 셀은 자신의 검증 후 실행합니다.


In [ ]:
# 정답 코드 실행 검증
assert stock_answer == {'노트': 3, '연필': 3}
assert [row['order_id'] for row in processed_answer] == ['A-101', 'A-103']
assert rejected_answer == [('A-102', '재고 부족'), ('A-104', '없는 상품')]
assert not orders_answer
print('solution_verification=PASS')

---

## 풀이 비교 원칙

각 참고 정답은 해당 실습의 학습자 검증 바로 뒤에 배치되어 있습니다. 정답 코드는 원래 실습 변수를 덮어쓰지 않도록 `_answer` 접미사를 사용합니다.

정답이 다르게 생겼더라도 **같은 결과와 처리 규칙** 을 만족하면 좋은 풀이입니다.

## 전체 마무리: 어떤 구조를 고를까?

- **리스트:** 순서 있는 데이터를 모으고 자유롭게 수정합니다.
- **튜플:** 항목의 의미와 개수가 고정된 묶음을 표현합니다.
- **딕셔너리:** 키로 값을 찾고 갱신합니다.
- **`deque`(선택/자습):** 단일 스레드에서 효율적인 FIFO 큐를 만듭니다.
- **`queue.Queue`(선택/자습):** 여러 스레드 사이에서 작업을 안전하게 전달합니다.

### 수업 핵심 종료 점검

1. `alias = original`과 `copied = original.copy()`의 차이를 설명할 수 있나요?
2. 튜플이 리스트보다 더 잘 맞는 고정 형식 데이터의 예를 들 수 있나요?
3. `d[key]`와 `d.get(key, default)`는 키가 없을 때 어떻게 다른가요?
4. 센서 보고서에서 리스트, 튜플, 딕셔너리가 각각 어떤 역할을 맡았나요?
5. 그룹 만들기와 요약 만들기를 두 함수로 나눈 이유는 무엇인가요?

### 선택/자습 종료 점검

6. 큐에 리스트의 `pop(0)` 대신 `deque.popleft()`를 쓰는 이유는 무엇인가요?
7. 서점 주문 처리에서 실패 주문이 재고를 바꾸면 안 되는 이유를 설명할 수 있나요?

학습자 검증은 자신의 TODO를, 접힌 정답 검증은 `_answer` 또는 `_solution` 참고 구현을 확인합니다.


In [ ]:
# 선택 사항: 핵심 개념 최종 자동 점검
from collections import deque

check_list = [1, 2]
check_copy = check_list.copy()
check_copy.append(3)

check_tuple = ("x", "y")
check_dict = {"known": 1}
check_queue = deque(["first", "second"])

assert check_list == [1, 2]
assert check_tuple[0] == "x"
assert check_dict.get("unknown", 0) == 0
assert check_queue.popleft() == "first"

print("최종 점검 통과: 네 자료구조의 기본 동작이 정상입니다.")